# <font color="brown">Visualizing Customer Segments (t-SNE vs. PCA) </font>

## <font color = "brown">Problem Statement </font>

### <font color="blue"> Context

The K-Means Clustering case study found 5 customer segments in the bank's 20,000-customer, 11-feature dataset, and visualized them by compressing down to 2 dimensions with PCA. That PCA plot worked reasonably well, but PCA is a *linear* technique, it can only stretch and rotate the data, never bend it. The analytics team wants to know: is there a better way to see how well-separated these segments really are, one that can capture bends and curves in the data that a straight-line technique like PCA would flatten out or miss?

### <font color="blue"> Objective

- Visualize the same 5 K-Means segments using t-SNE, a nonlinear visualization technique, and compare it directly against the PCA visualization already built.
- Understand practically how t-SNE's main setting, perplexity, changes the resulting picture, and why that makes t-SNE trickier to use than PCA.

### <font color="blue"> Data Dictionary

Same 20,000-customer dataset as the K-Means case study (Age, Tenure, Balance, Num_Products, Transactions_Per_Month, Digital_Engagement_Score, Credit_Utilization, and others), plus the Segment label (0-4) assigned by that case study's K-Means model.

## <font color="brown"> Importing Necessary Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

%matplotlib inline

## <font color="brown"> Recreating the K-Means Segments

We start exactly where the K-Means case study left off: same 11 features, same scaling, same `k=5`, same `random_state`, so the Segment labels here are identical to that notebook's.

In [ ]:
df = pd.read_csv(r"../KMeans Clustering/customer_segmentation.csv")
num_col = [c for c in df.columns if c != "Customer_ID"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[num_col])

kmeans = KMeans(n_clusters=5, random_state=1, n_init=10)
df["Segment"] = kmeans.fit_predict(X_scaled)
df["Segment"].value_counts().sort_index()

## <font color="brown"> A Fair Comparison Needs a Shared Subsample

t-SNE is far more computationally expensive than PCA, it does not scale comfortably to 20,000 points the way PCA does. The standard practice is to run it on a representative random subsample. To keep the comparison fair, we draw **one** subsample and use it for *both* the PCA and the t-SNE plots below, any visual difference between the two plots will then be about the technique, not about which points happened to be plotted.

In [ ]:
rng = np.random.default_rng(1)
sample_idx = rng.choice(len(X_scaled), size=4000, replace=False)

X_sample = X_scaled[sample_idx]
segment_sample = df["Segment"].values[sample_idx]
print("Subsample size:", X_sample.shape[0])
print(pd.Series(segment_sample).value_counts().sort_index())

## <font color="brown"> Baseline: PCA in 2D

This is the same technique, and nearly the same plot, as the K-Means case study, just recomputed on the shared subsample:

In [ ]:
pca = PCA(n_components=2, random_state=1)
pcs = pca.fit_transform(X_sample)

print("Variance captured by these 2 components:", round(pca.explained_variance_ratio_.sum(), 4))

fig, ax = plt.subplots(figsize=(8, 6.5))
scatter = ax.scatter(pcs[:, 0], pcs[:, 1], c=segment_sample, cmap='tab10', s=10, alpha=0.6)
legend1 = ax.legend(*scatter.legend_elements(), title='Segment', loc='best')
ax.add_artist(legend1)
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('Customer Segments in 2D (PCA)')
plt.show()

## <font color="brown"> t-SNE: Perplexity Changes the Picture

t-SNE's main setting is **perplexity**, loosely, how many neighbors each point tries to stay close to when the algorithm decides what "local structure" to preserve. Unlike PCA (which has no comparable knob), the same data can look meaningfully different at different perplexity values, so it's worth seeing that sensitivity directly rather than trusting a single default run.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
perplexities = [5, 30, 100]
embeddings = {}

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=1, init='pca', learning_rate='auto')
    emb = tsne.fit_transform(X_sample)
    embeddings[perp] = emb
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=segment_sample, cmap='tab10', s=10, alpha=0.6)
    ax.set_title(f'perplexity = {perp}')
    ax.set_xlabel('t-SNE dimension 1')
    ax.set_ylabel('t-SNE dimension 2')

plt.suptitle('The Same Data, Three Perplexity Values', fontsize=14)
plt.tight_layout()
plt.show()

**Low perplexity (5)** carves the data into many small, tightly-packed islands, it's paying attention to only a handful of nearest neighbors at a time, so it happily fragments what is really one segment into several visual clumps. **High perplexity (100)** does the opposite, it tries to respect a much larger neighborhood at once, and segments start to blur together or overlap more. **Perplexity 30** (roughly the commonly-used default, and a reasonable fraction of this sample size) gives the clearest, most stable-looking separation between segments here. There is no universally "correct" perplexity, unlike PCA's explained-variance percentage, t-SNE gives no single built-in number to optimize, checking a few values and looking for a stable picture is standard practice.

## <font color="brown"> Side-by-Side: PCA vs. t-SNE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

sc0 = axes[0].scatter(pcs[:, 0], pcs[:, 1], c=segment_sample, cmap='tab10', s=10, alpha=0.6)
axes[0].set_title('PCA (linear)')
axes[0].set_xlabel('PC 1'); axes[0].set_ylabel('PC 2')

best_emb = embeddings[30]
sc1 = axes[1].scatter(best_emb[:, 0], best_emb[:, 1], c=segment_sample, cmap='tab10', s=10, alpha=0.6)
axes[1].set_title('t-SNE (nonlinear), perplexity=30')
axes[1].set_xlabel('t-SNE dim 1'); axes[1].set_ylabel('t-SNE dim 2')

legend1 = axes[1].legend(*sc1.legend_elements(), title='Segment', loc='best')
axes[1].add_artist(legend1)
plt.suptitle('Same 5 Segments, Two Different Ways of Looking at Them', fontsize=14)
plt.tight_layout()
plt.show()

Both plots agree on the headline finding: 5 segments, genuinely separable, matching what the K-Means case study's silhouette analysis already established. But the shapes differ. In the PCA plot, segments that overlap along the two straight axes it's allowed to draw show up as blended, smeared regions. t-SNE, free to bend and curve its layout, pulls those same points apart into cleaner, rounder clusters, because it is only trying to preserve *who is near whom*, not the actual straight-line distances or directions in the original 11-dimensional space.

## <font color="brown"> The Catch: What t-SNE Distances Do *Not* Mean

It's tempting to read a t-SNE plot the same way as a PCA plot, treating distance and cluster size at face value. That instinct is often wrong for t-SNE specifically:

In [ ]:
# Distances between cluster centers in PCA space are meaningful (they're literally rotated/
# projected original distances). Check whether the same is true in t-SNE space.
pca_centers = pd.DataFrame(pcs, columns=['x', 'y']).assign(Segment=segment_sample).groupby('Segment').mean()
tsne_centers = pd.DataFrame(best_emb, columns=['x', 'y']).assign(Segment=segment_sample).groupby('Segment').mean()

from scipy.spatial.distance import pdist, squareform
print("Pairwise PCA center distances:\n", squareform(pdist(pca_centers)).round(2))
print("\nPairwise t-SNE center distances:\n", squareform(pdist(tsne_centers)).round(2))

The two distance tables don't agree with each other, and that's expected, not a bug. t-SNE's objective only tries to keep genuine neighbors close together locally; it makes no promise about how far apart two different clusters end up, or how large a cluster's plotted area is. A cluster that looks huge in a t-SNE plot isn't necessarily more spread out in reality than a cluster that looks tiny, t-SNE can pack or stretch clusters somewhat arbitrarily to make the layout work. **Never read cluster size or inter-cluster distance off a t-SNE plot as if it were to scale.** The one thing a t-SNE plot is built to get right is which points are close neighbors of which.

## <font color="brown"> Business Insights and Recommendations

- **Both PCA and t-SNE confirm the same underlying finding**: the K-Means case study's 5 segments are genuinely separable groups, not an artifact of forcing `k=5`, this cross-check with an entirely different visualization technique is reassuring.

- **Use PCA first, by default.** It's fast enough to run on the full 20,000 customers, its axes have a real, interpretable meaning (it's the same technique from the PCA case study), and distances on the plot genuinely reflect distances in the original data.

- **Reach for t-SNE specifically when the PCA plot looks ambiguous** (clusters visibly overlapping or smeared together) and the actual question is *are these truly separate groups, or just one group viewed from a bad angle?* t-SNE's ability to bend the layout can reveal separation that a straight-line projection is structurally unable to show.

- **Treat t-SNE as a visualization tool only, never as a feature-engineering or distance-measurement step.** Its axes have no fixed meaning (they can flip or rotate between runs), and, as shown above, distances between clusters are not to scale. PCA remains the right tool whenever actual distances, loadings, or reproducible axes are needed downstream.

- **Always check a couple of perplexity values before trusting a t-SNE plot.** A single run at an unexamined default risks either an over-fragmented picture (perplexity too low) or an over-merged one (too high).